# TCAD: Baehr-Jones 2012 silicon traveling-wave modulator

This notebook reproduces the device-physics half of Baehr-Jones *et al.*,
"Ultralow drive voltage silicon traveling-wave modulator", *Optics Express*
**20**(11), 12014 (2012), [doi:10.1364/OE.20.012014](https://doi.org/10.1364/OE.20.012014),
with generic gsim tools. DEVSIM solves the lateral PN junction in the rib,
femwell turns its carriers into a phase shift and an optical loss, and a
closed-form transmission-line model loads the electrode with the junction
capacitance. The metal-only electrode is the subject of the
[Palace electrode notebook](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_bj2012_electrode.ipynb).

Reference values (read by eye from the paper's figures, about one marker of
uncertainty) are in `data/bj2012_reference.json`.

| Step | Question | Tool | Result |
|---|---|---|---|
| 1. Junction | How does the junction capacitance change with bias? | DEVSIM, `gsim.tcad` | $C(V)$ against the paper's Fig. 2d |
| 2. Phase shifter | What do the carriers do to the guided mode? | femwell, `gsim.common.carriers` | $\Delta n_{eff}(V)$, $V_\pi L$ (Fig. 2c), implant loss, group index |
| 3. Loaded line | What does the junction do to the electrode? | closed form, `gsim.common.transmission_line` and `gsim.common.twmzm` | $Z_0$, RF index, RF loss, electro-optic 3 dB bandwidth |

**Fit and prediction.** The paper does not give the doping profile. Step 1
fits one number, the lateral grading $\sigma$ of the junction, to the
capacitance in Fig. 2d. The results of Steps 2 and 3 are predictions that
follow from that fit. The Summary table marks every row.

**Requirements:** the `tcad` and `femwell` extras, `pip install "gsim[tcad,femwell]"`.
Everything runs locally and nothing is submitted to a cloud service.

In [ ]:
import json
import logging
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np

from gsim.common.cross_section import build_doped_cross_section
from gsim.common.stack.extractor import Layer
from gsim.common.stack.materials import MATERIALS_DB, make_doped_materials

# The mesher's advice on air boxes targets 3D driven simulations; it does not
# apply to these cross-section solves.
logging.getLogger("gsim.palace").setLevel(logging.ERROR)

REF_PATH = next(
    p
    for p in (
        Path("data/bj2012_reference.json"),
        Path("nbs/data/bj2012_reference.json"),
    )
    if p.exists()
)
REF = json.loads(REF_PATH.read_text(encoding="utf-8"))

## The device

The cross-section follows the paper's Fig. 1b: a 220 nm silicon rib, 0.5 µm
wide, on a 90 nm slab, with the PN junction in the middle of the rib. The
heavily doped contact regions start 0.95 µm from the rib edge and reach the
3.55 µm mark, where the 0.75 µm thick aluminum lands on them. The cell below
draws each doped region on a layer of its own and builds the layer stack the
cross-section is cut from.

The paper does not state the contact doping (1e20 cm⁻³ is assumed) or the
junction position (the rib centre is assumed). Its quoted peak doping, p 7e17
and n 5e17 cm⁻³, is used as stated first. The slab and contact doping follow
in Step 1.

**Striations.** The paper's implants are segmented along the light's direction
(Fig. 1a inset: 10 µm period, 1 µm gaps), so only `FILL` = 90% of the device
length carries a junction. The simulation is a 2D cross-section *of a
segment*, while the measured quantities (capacitance and series resistance per
unit length, $V_\pi L$, loss) are averages over the whole length. Every
per-length result is therefore multiplied by `FILL` before it meets the paper:
capacitance, $\Delta n_{eff}$ and loss. The series resistance is divided by
`FILL`, so that $R_s C_j$ stays that of a segment.

In [ ]:
# Lengths in um, from the reference file.
WAVELENGTH_UM = REF["geometry"]["wavelength_um"]
CENTER_Y = -20.0  # junction position along y (um)
RIB_HEIGHT = REF["geometry"]["rib_height_um"]
SLAB_HEIGHT = REF["geometry"]["slab_height_um"]
RIB_HALF = REF["geometry"]["rib_width_um"] / 2
SLAB_WIDTH = REF["geometry"]["contact_start_from_rib_edge_um"]
CONTACT_WIDTH = (
    REF["geometry"]["rib_edge_to_via_side_um"]
    - REF["geometry"]["contact_start_from_rib_edge_um"]
)
METAL_THICKNESS = REF["geometry"]["lower_metal_thickness_um"]
# Fraction of the device length that carries a junction (Fig. 1a inset).
FILL = (
    1
    - REF["geometry"]["implant_striation_gap_um"]
    / REF["geometry"]["implant_striation_period_um"]
)

# Regions from the n-side contact to the p-side one:
# name: (dopant, width um, height um)
REGIONS = {
    "n_contact": ("donor", CONTACT_WIDTH, SLAB_HEIGHT),
    "n_slab": ("donor", SLAB_WIDTH, SLAB_HEIGHT),
    "n_rib": ("donor", RIB_HALF, RIB_HEIGHT),
    "p_rib": ("acceptor", RIB_HALF, RIB_HEIGHT),
    "p_slab": ("acceptor", SLAB_WIDTH, SLAB_HEIGHT),
    "p_contact": ("acceptor", CONTACT_WIDTH, SLAB_HEIGHT),
}
ELECTRODES = {"n_contact": "cathode_metal", "p_contact": "anode_metal"}

gf.gpdk.PDK.activate()
component = gf.Component()
layers, spans = {}, {}
y = CENTER_Y - sum(width for _, width, _ in REGIONS.values()) / 2
for i, (name, (_, width, height)) in enumerate(REGIONS.items()):
    spans[name] = (y, y + width)
    rect = component << gf.c.rectangle((10.0, width), layer=(30, i))
    rect.y = y + width / 2
    layers[name] = Layer(
        name=name,
        gds_layer=(30, i),
        zmin=0.0,
        zmax=height,
        thickness=height,
        material=name,
        layer_type="dielectric",
        mesh_resolution="medium" if "contact" in name else "fine",
    )
    y += width
for i, (region, metal) in enumerate(ELECTRODES.items()):
    y0, y1 = spans[region]
    rect = component << gf.c.rectangle((10.0, y1 - y0), layer=(46, i))
    rect.y = (y0 + y1) / 2
    layers[metal] = Layer(
        name=metal,
        gds_layer=(46, i),
        zmin=SLAB_HEIGHT,
        zmax=SLAB_HEIGHT + METAL_THICKNESS,
        thickness=METAL_THICKNESS,
        material="aluminum",
        layer_type="conductor",
        mesh_resolution="fine",
    )

materials = make_doped_materials([(name, 0.0) for name in REGIONS], permittivity=11.9)
sellmeier = next(
    m for m in MATERIALS_DB["silicon"].dispersion_models if m.type == "sellmeier"
)
for properties in materials.values():
    properties.dispersion_models.append(sellmeier)
stack, _ = build_doped_cross_section(
    component,
    axis="x",
    value=0.0,
    substrate_thickness=2.0,
    doping={"layer_specs": layers, "materials": materials},
    verbose=False,
)

DOPED_SPAN = (spans["n_contact"][0], spans["p_contact"][1])
RIB_SPAN = (spans["n_rib"][0], spans["p_rib"][1])
CHARGE_WINDOW = (DOPED_SPAN[0] - 0.5, DOPED_SPAN[1] + 0.5)
OPTICAL_WINDOW = (CENTER_Y - 2.0, CENTER_Y + 2.0)
OPTICAL_WINDOW_Z = (-1.0, RIB_HEIGHT + 1.0)
JUNCTION_BOX = (*RIB_SPAN, 0.0, RIB_HEIGHT, 0.01)  # (y0, y1, z0, z1, size) um

for name, (dopant, width, height) in REGIONS.items():
    y0, y1 = spans[name]
    print(
        f"{name:10s} y = {y0:7.2f} .. {y1:7.2f} um, {width:4.2f} um wide, "
        f"{height * 1e3:3.0f} nm tall, {dopant}"
    )

The helper below draws a cross-section mesh with one colour per region. It is
plotting code only.

In [ ]:
import meshio
from matplotlib.collections import PolyCollection

REGION_COLORS = {}  # one colour per region name, stable across figures


def plot_cross_section(mesh_path, *, ax, title=None, xlim=None, ylim=None):
    """Draw a cross-section mesh with one colour per region."""
    mesh = meshio.read(mesh_path, file_format="gmsh")
    points = mesh.points[:, :2]
    names = {tag: name for name, (tag, dim) in mesh.field_data.items() if dim == 2}
    triangles = mesh.cells_dict["triangle"]
    tags = mesh.cell_data_dict["gmsh:physical"]["triangle"]
    handles = []
    for tag in sorted(set(tags), key=names.get):
        name = names[tag]
        color = REGION_COLORS.setdefault(
            name, plt.get_cmap("tab20")(len(REGION_COLORS) % 20)
        )
        ax.add_collection(
            PolyCollection(
                points[triangles[tags == tag]],
                facecolors=color,
                edgecolors="k",
                linewidths=0.1,
            )
        )
        handles.append(plt.Rectangle((0, 0), 1, 1, color=color, label=name))
    ax.autoscale_view()
    ax.set_aspect("equal")
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set(xlabel="y (um)", ylabel="z (um)", title=title)
    ax.legend(
        handles=handles,
        fontsize=7,
        ncol=min(len(handles), 6),
        loc="upper center",
        bbox_to_anchor=(0.5, -0.3),
    )
    return ax

## Step 1: junction capacitance (Fig. 2d)

`gsim.tcad` solves Poisson and drift-diffusion on the doped silicon at every
bias and takes the capacitance per unit length from a small-signal AC solve.
Each metal on a contact region is a **contact**, every pair of touching doped
regions an **interface**, and the oxide around the silicon an **insulator**.

`build_charge_sim` takes any list of doping profiles, so the as-stated profile
and the calibrated one below share the same geometry and mesh settings.

In [ ]:
from itertools import pairwise

from gsim import tcad


def build_charge_sim(profiles, *, output_dir, refined_um=0.02, junction_box_um=0.01):
    """A meshed `tcad.Simulation` of the junction with the given doping profiles."""
    sim = tcad.Simulation()
    sim.geometry(component, stack=stack)
    sim.domain(
        plane="x=0",
        window=CHARGE_WINDOW,
        margin_x=2.0,
        margin_y=2.0,
        z_above=1.5,
        z_below=1.0,
    )
    sim.materials(insulators={"sio2": stack.materials["sio2"]["permittivity"]})
    for profile in profiles:
        sim.doping.add(profile)
    sim.contacts.add(
        name="cathode", semiconductor="n_contact", electrode="cathode_metal"
    )
    sim.contacts.add(name="anode", semiconductor="p_contact", electrode="anode_metal")
    for left, right in pairwise(REGIONS):
        junction = (left, right) == ("n_rib", "p_rib")
        sim.interfaces.add(
            name="junction" if junction else f"{left}_{right}",
            region_a=left,
            region_b=right,
        )
    sim.solver(
        mesh_preset="coarse",
        refined_mesh_size_um=refined_um,
        max_mesh_size_um=40.0,
        refinement_boxes=[(*RIB_SPAN, 0.0, RIB_HEIGHT, junction_box_um)],
    )
    mesh = sim.mesh(output_dir=output_dir)
    return sim, mesh


def uniform_profiles(concentration_cm3):
    """One uniform StepDoping per region from a {region: cm^-3} dict."""
    return [
        tcad.StepDoping(
            region=name, dopant_type=REGIONS[name][0], concentration_cm3=value
        )
        for name, value in concentration_cm3.items()
    ]

In [ ]:
# The paper's quoted peak doping (p 7e17, n 5e17 cm^-3) taken as uniform over
# rib and slab, with 1e20 contacts.
P_PEAK, N_PEAK = REF["doping"]["p_peak_cm3"], REF["doping"]["n_peak_cm3"]
STATED = {
    "n_contact": 1e20,
    "n_slab": N_PEAK,
    "n_rib": N_PEAK,
    "p_rib": P_PEAK,
    "p_slab": P_PEAK,
    "p_contact": 1e20,
}
stated_sim, stated_mesh = build_charge_sim(
    uniform_profiles(STATED), output_dir="./tcad-bj2012/stated"
)

The top panel shows the whole charge window and the bottom panel the junction
at the rib centre. Look for the element size across the rib, where the
depletion edges move with bias.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), height_ratios=(1, 2))
plot_cross_section(
    stated_mesh.mesh_path,
    ax=axes[0],
    title="Charge window: the doped slab",
    ylim=(-0.3, 1.2),
)
axes[0].get_legend().remove()  # the close-up below carries it
plot_cross_section(
    stated_mesh.mesh_path,
    ax=axes[1],
    title="Around the junction",
    xlim=(CENTER_Y - 0.6, CENTER_Y + 0.6),
    ylim=(-0.05, 0.3),
)
plt.tight_layout()
plt.show()

The elements are 10 nm across the rib (the refinement box) and coarser in the
contact regions, where nothing changes quickly.

### As-stated doping

The first run uses the profile exactly as quoted: uniform doping at the paper's
peak values in rib and slab. The sweep takes a few minutes.

In [ ]:
CV_BIASES = np.array(
    REF["capacitance"]["bias_v_reverse"]
)  # V on the cathode = reverse bias
CV_PAPER_FF_UM = np.array(REF["capacitance"]["c_fF_per_um"])
# 1 F/cm of depth = 1e15 fF / 1e4 um = 1e11 fF/um.
F_PER_CM_TO_FF_PER_UM = 1e11

In [ ]:
stated_sweep = stated_sim.sweep(CV_BIASES, contact="cathode")

The next figure compares the simulated capacitance, scaled by the striation
fill, with the paper's markers and with the abrupt-junction formula (Sze) for
the same doping. Look at the gap between the simulation and the markers.

In [ ]:
from gsim.common.stack.pn_junction import PNJunctionConfig

junction = PNJunctionConfig(na_cm3=P_PEAK, nd_cm3=N_PEAK)
stated_cmp = tcad.compare_capacitance(junction, stated_sweep, height_um=RIB_HEIGHT)
# Capacitance per unit length of *device*: the segment's value times FILL.
stated_ff_um = FILL * stated_cmp.c_tcad_f_per_cm * F_PER_CM_TO_FF_PER_UM
sze_ff_um = FILL * stated_cmp.c_analytic_f_per_cm * F_PER_CM_TO_FF_PER_UM

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(CV_BIASES, CV_PAPER_FF_UM, "ks", label="paper, Fig. 2d (read by eye)")
ax.plot(CV_BIASES, stated_ff_um, "o-", label="TCAD, doping as stated")
ax.plot(CV_BIASES, sze_ff_um, "^--", label="abrupt-junction formula (Sze)")
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C per device length (fF/um)")
ax.set_title("Junction capacitance, as-stated doping")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("  V    TCAD   Sze   paper  (fF/um)  TCAD/paper")
for v, c, s, p in zip(CV_BIASES, stated_ff_um, sze_ff_um, CV_PAPER_FF_UM, strict=True):
    print(f"{v:5.2f}  {c:5.3f}  {s:5.3f}  {p:5.3f}      {c / p:4.2f}")

At 0 V the simulated capacitance is 1.9 times the paper's, and at 4 V it is
1.5 times. The abrupt-junction formula for the same doping is also above the
paper's up to 2 V, so the quoted peak doping alone does not reproduce Fig. 2d.

Before calibrating, the next cell checks the solver against the formula where
the formula applies: the depletion width along a horizontal cut through the
middle of the rib.

In [ ]:
def depletion_width_um(point, *, na_cm3, nd_cm3):
    """Depletion width along the mid-rib lateral cut of one bias point."""
    carriers = point.carriers
    cut = (np.abs(carriers.y_um - RIB_HEIGHT / 2) < 0.03) & (
        (carriers.x_um >= RIB_SPAN[0]) & (carriers.x_um <= RIB_SPAN[1])
    )
    order = np.argsort(carriers.x_um[cut])
    return tcad.estimate_depletion_width_um(
        carriers.x_um[cut][order],
        carriers.electrons_cm3[cut][order],
        carriers.holes_cm3[cut][order],
        na_cm3=na_cm3,
        nd_cm3=nd_cm3,
    )


for v in (0.0, 2.0):
    point = stated_sweep.points[int(np.argmin(np.abs(CV_BIASES - v)))]
    w_tcad = depletion_width_um(point, na_cm3=P_PEAK, nd_cm3=N_PEAK)
    w_sze = PNJunctionConfig(na_cm3=P_PEAK, nd_cm3=N_PEAK, v_reverse=v).w_um
    print(f"V = {v:g} V: depletion width TCAD {w_tcad:.3f} um, formula {w_sze:.3f} um")

The solver's depletion width is 8 to 9% below the formula's (0.059 against
0.064 µm at 0 V, 0.104 against 0.114 µm at 2 V), so the two agree where the
formula applies. The capacitance gap points to the doping profile, not to the
solver.

### Slab doping from the sheet resistance

The paper quotes the sheet resistance of the partially etched slab, 10 kΩ/sq
(p) and 7 kΩ/sq (n). It fixes the slab doping independently of the rib's peak
values through $R_\square = 1 / (q \mu(N) N t)$, with the 90 nm slab thickness
$t$ and the Masetti mobility model that gsim's TCAD uses.

In [ ]:
from scipy.optimize import brentq

from gsim.common.carriers import MobilityModel

Q_C = 1.602176634e-19
MOBILITY = MobilityModel.masetti_silicon()


def sheet_resistance_ohm_sq(concentration_cm3, mobility_cm2, thickness_um):
    """Sheet resistance of a uniformly doped, fully ionised film."""
    return 1.0 / (
        Q_C * mobility_cm2(concentration_cm3) * concentration_cm3 * thickness_um * 1e-4
    )


SHEET_P = REF["doping"]["slab_sheet_resistance_p_kohm_per_sq"] * 1e3
SHEET_N = REF["doping"]["slab_sheet_resistance_n_kohm_per_sq"] * 1e3
SLAB_P_CM3 = brentq(
    lambda c: sheet_resistance_ohm_sq(c, MOBILITY.holes_cm2, SLAB_HEIGHT) - SHEET_P,
    1e16,
    1e19,
)
SLAB_N_CM3 = brentq(
    lambda c: sheet_resistance_ohm_sq(c, MOBILITY.electrons_cm2, SLAB_HEIGHT) - SHEET_N,
    1e16,
    1e19,
)
print(
    f"p slab: {SLAB_P_CM3:.2e} cm^-3 -> {sheet_resistance_ohm_sq(SLAB_P_CM3, MOBILITY.holes_cm2, SLAB_HEIGHT) / 1e3:.2f} kohm/sq"
    f" (mu_p = {float(MOBILITY.holes_cm2(SLAB_P_CM3)):.0f} cm^2/Vs)"
)
print(
    f"n slab: {SLAB_N_CM3:.2e} cm^-3 -> {sheet_resistance_ohm_sq(SLAB_N_CM3, MOBILITY.electrons_cm2, SLAB_HEIGHT) / 1e3:.2f} kohm/sq"
    f" (mu_n = {float(MOBILITY.electrons_cm2(SLAB_N_CM3)):.0f} cm^2/Vs)"
)

The slab doping is 2.8e17 cm⁻³ (p) and 1.6e17 cm⁻³ (n).

### What the paper's own curve says about the profile

The slab doping does not move the capacitance; it sets $R_s$. The mismatch is
in the rib, and the shape of Fig. 2d says how. The next cell fits
$C = C_0 + K\,(V_{bi} + V)^{-m}$ to the digitised points with $V_{bi} = 0.8$ V.
An abrupt junction has $m = 1/2$, a linearly graded one $m = 1/3$.

In [ ]:
from scipy.optimize import curve_fit


def cv_law(v, c0, k, m, v_bi=0.8):
    """C0 + K (V_bi + V)^-m, the depletion law of a graded junction plus an offset."""
    return c0 + k * (v_bi + v) ** (-m)


popt, _ = curve_fit(cv_law, CV_BIASES, CV_PAPER_FF_UM, p0=(0.05, 0.16, 0.4))
fit_rms = float(np.sqrt(np.mean((cv_law(CV_BIASES, *popt) - CV_PAPER_FF_UM) ** 2)))
print(
    f"paper C(V): C0 = {popt[0]:.3f} fF/um, K = {popt[1]:.3f}, exponent m = {popt[2]:.2f} (rms {fit_rms * 1e3:.1f} aF/um)"
)
print("abrupt junction m = 0.50, linearly graded m = 0.33")

The fit gives $m = 0.36$ (rms 0.8 aF/µm), close to the graded value of 1/3.
The measured junction behaves like a graded one, as lateral implant diffusion
makes it.

### A graded junction, fitted to Fig. 2d

The model keeps everything the paper states (the 7e17 and 5e17 cm⁻³ peaks, the
slab doping from the sheet resistance and 1e20 cm⁻³ contacts) and replaces the
abrupt step at the rib centre by complementary-error-function tails of both
implants,

$$N_D(y) = \tfrac{1}{2} N_n \,\mathrm{erfc}\!\left(\frac{y-y_j}{\sqrt{2}\,\sigma}\right), \qquad
N_A(y) = \tfrac{1}{2} N_p \,\mathrm{erfc}\!\left(-\frac{y-y_j}{\sqrt{2}\,\sigma}\right),$$

with the junction at the rib centre $y_j$ and one free parameter, the lateral
straggle $\sigma$.

**$\sigma$ is the only fitted number.** The value $\sigma$ = 0.27 µm brings
$C(V)$ within 10% of Fig. 2d with the striation fill applied, that is, with the
segment's capacitance at $1/0.9$ times the paper's. The capacitance falls
monotonically with $\sigma$, as the sensitivity cells below show. Agreement
with Fig. 2d therefore only shows that the fit worked. The predictions are in
Steps 2 and 3.

In [ ]:
from scipy.special import erfc

SIGMA_UM = 0.27  # the fitted lateral straggle (um)


def graded_profiles(sigma_um):
    """Erfc-graded n and p implants across the rib; slab and contact doping as above."""
    junction_y = CENTER_Y
    plateau = {  # region: (donor plateau, acceptor plateau) in cm^-3
        "n_rib": (N_PEAK, P_PEAK),
        "p_rib": (N_PEAK, P_PEAK),
        "n_slab": (SLAB_N_CM3, SLAB_P_CM3),
        "p_slab": (SLAB_N_CM3, SLAB_P_CM3),
    }
    profiles = []
    for region, (n_plateau, p_plateau) in plateau.items():
        profiles.append(
            tcad.CallableDoping(
                region=region,
                dopant_type="donor",
                function=lambda y, _z, nd=n_plateau: (
                    nd * 0.5 * erfc((y - junction_y) / (np.sqrt(2) * sigma_um))
                ),
            )
        )
        profiles.append(
            tcad.CallableDoping(
                region=region,
                dopant_type="acceptor",
                function=lambda y, _z, na=p_plateau: (
                    na * 0.5 * erfc(-(y - junction_y) / (np.sqrt(2) * sigma_um))
                ),
            )
        )
    profiles.extend(
        tcad.StepDoping(
            region=region, dopant_type=REGIONS[region][0], concentration_cm3=1e20
        )
        for region in ("n_contact", "p_contact")
    )
    return profiles

In [ ]:
graded_sim, _ = build_charge_sim(
    graded_profiles(SIGMA_UM), output_dir="./tcad-bj2012/graded"
)

### Sweep the calibrated junction

One sweep serves the rest of the notebook. It covers the Fig. 2d biases, the
Fig. 2c biases and -0.5 V, so that the slope at the most forward Fig. 2c bias
(-0.4 V) comes from points on both sides of it.

In [ ]:
VPIL_BIASES = np.array(REF["vpil_V_cm"]["bias_v_reverse"])
SWEEP_BIASES = np.unique(np.concatenate([CV_BIASES, VPIL_BIASES, [-0.5]]))
print(f"{len(SWEEP_BIASES)} biases from {SWEEP_BIASES[0]:g} to {SWEEP_BIASES[-1]:g} V")

In [ ]:
graded_sweep = graded_sim.sweep(SWEEP_BIASES, contact="cathode")

The next figure adds the calibrated curve to the paper's markers and to the
as-stated curve. Look for the calibrated curve passing through the markers at
every bias.

In [ ]:
graded_v = graded_sweep.voltages
graded_cmp = tcad.compare_capacitance(junction, graded_sweep, height_um=RIB_HEIGHT)
graded_ff_um = FILL * graded_cmp.c_tcad_f_per_cm * F_PER_CM_TO_FF_PER_UM
at_paper = np.array([int(np.argmin(np.abs(graded_v - v))) for v in CV_BIASES])
graded_at_paper = graded_ff_um[at_paper]

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(CV_BIASES, CV_PAPER_FF_UM, "ks", label="paper, Fig. 2d (read by eye)")
ax.plot(
    CV_BIASES,
    stated_ff_um,
    "o--",
    color="C0",
    alpha=0.6,
    label="TCAD, doping as stated",
)
ax.plot(
    graded_v,
    graded_ff_um,
    "o-",
    color="C3",
    label=f"TCAD, graded junction (sigma = {SIGMA_UM} um, fitted)",
)
ax.plot(CV_BIASES, sze_ff_um, "^:", color="gray", label="abrupt-junction formula (Sze)")
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C per device length (fF/um)")
ax.set_title("Junction capacitance: as stated vs calibrated")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

ratio = graded_at_paper / CV_PAPER_FF_UM
print("  V    graded  paper  graded/paper")
for v, c, p, r in zip(CV_BIASES, graded_at_paper, CV_PAPER_FF_UM, ratio, strict=True):
    print(f"{v:5.2f}  {c:5.3f}  {p:5.3f}   {r:4.2f}")
print(
    f"largest deviation from the paper: {np.max(np.abs(ratio - 1)):.1%} (target +/-10%; a fit, not a prediction)"
)

The calibrated curve stays within 3.1% of Fig. 2d at all ten biases (target
±10%), where the as-stated profile is 1.5 to 2.0 times too high. The curve is
fitted, so the agreement shows that the calibration worked and not that the
physics is predicted.

### Sensitivity to the fitted number

The same curve for σ = 0.22 and 0.32 µm shows what the uncertainty of the fit
is worth. The runs use a coarser mesh to keep them short (0.04 µm elements,
20 nm in the rib instead of 0.02 µm and 10 nm), and the fitted σ is solved on
the same coarse mesh for comparison. The coarse mesh moves $C$ by about 1%.

In [ ]:
BAND_SIGMAS = (0.22, 0.27, 0.32)
BAND_BIASES = np.array([-0.4, 0.0, 2.0, 4.0])

In [ ]:
band = {}
for sigma in BAND_SIGMAS:
    sim, _ = build_charge_sim(
        graded_profiles(sigma),
        output_dir=f"./tcad-bj2012/band-{sigma:g}",
        refined_um=0.04,
        junction_box_um=0.02,
    )
    band[sigma] = (
        FILL
        * sim.sweep(BAND_BIASES, contact="cathode").capacitance_f_per_cm
        * F_PER_CM_TO_FF_PER_UM
    )

The shaded band in the next figure spans σ = 0.22 to 0.32 µm. Look at its
width against the paper's markers.

In [ ]:
band_paper = np.interp(BAND_BIASES, CV_BIASES, CV_PAPER_FF_UM)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.fill_between(
    BAND_BIASES,
    band[BAND_SIGMAS[0]],
    band[BAND_SIGMAS[-1]],
    color="C3",
    alpha=0.2,
    label=f"sigma = {BAND_SIGMAS[0]} to {BAND_SIGMAS[-1]} um",
)
ax.plot(
    BAND_BIASES,
    band[SIGMA_UM],
    "o-",
    color="C3",
    label=f"sigma = {SIGMA_UM} um (coarse mesh)",
)
ax.plot(BAND_BIASES, band_paper, "ks", label="paper, Fig. 2d")
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C per device length (fF/um)")
ax.set_title("Sensitivity of C(V) to the junction grading sigma")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()
for sigma in BAND_SIGMAS:
    print(
        f"sigma = {sigma:.2f} um: C/paper = {np.round(band[sigma] / band_paper, 2)} at {BAND_BIASES} V"
    )

At 0 V the band runs from 1.05 to 0.96 times the paper's capacitance, so a
change of ±0.05 µm in σ is worth about ±5%. The 4 V point drops away at large
σ (0.92 at 0.32 µm), where the depletion region fills the whole rib.

## Step 2: phase shifter (Fig. 2c) and implant loss

Free carriers lower the refractive index and absorb light (plasma dispersion).
Each bias point's carrier map becomes a material property through the
Soref-Bennett fit, with the Nedeljkovic fit as a comparison. The optical window
is meshed with the same native-2D mesher, the carrier map is carried onto it
element by element, and femwell solves for the guided mode. Each bias gives one
complex $n_{eff}$, and two results follow:

- **Phase shift.** $\Delta n_{eff}(V) = n_{eff}(V) - n_{eff}(0)$ gives
  $V_\pi L = \lambda / (2\,|d\Delta n_{eff}/dV|)$ from the local slope. The
  paper's Fig. 2c is "the small-signal $V_\pi L$ value" per arm, calculated
  from the phase shift of that arm against its bias, so the local slope is the
  right reading. A check after the $V_\pi L$ plot compares it with the chord
  value from 0 V at 4.5 V.
- **Loss.** $\alpha_{dB/cm} = (10/\ln 10)\,2 k_0\,|\mathrm{Im}\, n_{eff}|$ with
  $k_0 = 2\pi/\lambda$ is the power loss from the field decay.

Both are values for one segment. The device average is `FILL` times them,
because the gaps carry no carriers and so no phase shift and no free-carrier
loss. The segment's $V_\pi L$ becomes $V_\pi L / $`FILL`.

Nothing in this step is tuned, but it inherits the fitted doping profile of
Step 1, so every result here is a prediction.

In [ ]:
from gsim.common.carrier_transfer import transfer_carriers
from gsim.common.carriers import (
    PlasmaDispersionModel,
    carrier_absorption_cm,
    carrier_index_shift,
    permittivity_perturbation,
)
from gsim.common.modes import select_line_mode
from gsim.femwell import boundary_field_ratio, epsilon_by_region, solve_modes
from gsim.palace import BoundaryModeSim

DISPERSION_MODELS = {
    "Soref-Bennett": PlasmaDispersionModel.soref_1550(),
    "Nedeljkovic": PlasmaDispersionModel.nedeljkovic_1550(),
}

The optical window is meshed in oxide, with a refinement box over the rib.

In [ ]:
def cross_section_mesh(
    component, stack, *, plane, output_dir, window=None, window_z=None, boxes=None
):
    """Mesh a cross-section of a component in oxide and return the mesh path."""
    sim = BoundaryModeSim()
    sim.set_output_dir(output_dir)
    sim.set_stack(stack)
    sim.set_geometry(component)
    sim.set_airbox(
        margin_x=2.0, margin_y=2.0, z_above=1.5, z_below=1.0, material="sio2"
    )
    sim.set_cross_section(plane, window=window, window_z=window_z)
    sim.mesh(
        preset="coarse",
        refined_mesh_size=0.05,
        max_mesh_size=40.0,
        refinement_boxes=boxes,
        verbose=False,
    )
    return sim.mesh_path


optical_mesh = cross_section_mesh(
    component,
    stack,
    plane="x=0",
    output_dir="./tcad-bj2012/optical",
    window=OPTICAL_WINDOW,
    window_z=OPTICAL_WINDOW_Z,
    boxes=[JUNCTION_BOX],
)

`optical_mode` puts one carrier map on the optical mesh and solves for the
guided mode. `solve_optical_mode` returns its complex $n_{eff}$.

In [ ]:
def optical_mode(carriers, dispersion, wavelength_um=WAVELENGTH_UM):
    """Guided mode with one carrier map on the optical mesh.

    Returns the femwell mode, the carriers on the optical mesh, the absorption
    (cm^-1) and the complex permittivity per element.
    """
    base = epsilon_by_region(optical_mesh, stack, wavelength_um=wavelength_um)
    doped = [name for name in REGIONS if name in base]
    on_mesh = transfer_carriers(
        carriers, optical_mesh, at="elements", fill=0.0, regions=doped
    )
    index_shift = carrier_index_shift(
        on_mesh.electrons_cm3, on_mesh.holes_cm3, model=dispersion
    )
    absorption_cm = carrier_absorption_cm(
        on_mesh.electrons_cm3, on_mesh.holes_cm3, model=dispersion
    )
    epsilon = np.array([base[name] for name in on_mesh.region], dtype=complex)
    for i, name in enumerate(on_mesh.region):
        if name in doped:
            epsilon[i] = permittivity_perturbation(
                n0=np.sqrt(base[name].real),
                dn=index_shift[i],
                dalpha_cm=absorption_cm[i],
                wavelength_um=wavelength_um,
            )
    modes = solve_modes(
        optical_mesh,
        epsilon=epsilon,
        wavelength_um=wavelength_um,
        metallic_boundaries=True,
    )
    mode = select_line_mode(modes)
    assert boundary_field_ratio(mode) < 0.01, "the optical window clips the mode"
    return mode, on_mesh, absorption_cm, epsilon


def solve_optical_mode(carriers, dispersion, wavelength_um=WAVELENGTH_UM):
    """Complex n_eff of the guided mode with one carrier map on the optical mesh."""
    return complex(optical_mode(carriers, dispersion, wavelength_um)[0].n_eff)

The next cell solves the mode at every bias for both dispersion models. It
converts $n_{eff}$ into the device-average loss and index shift.

In [ ]:
n_eff = {
    label: np.array(
        [solve_optical_mode(point.carriers, model) for point in graded_sweep.points]
    )
    for label, model in DISPERSION_MODELS.items()
}
# |Im n_eff| is an amplitude decay: 4 pi |Im n_eff| / lambda per unit length in power.
LOSS_DB_PER_CM_PER_IMAG = 40 * np.pi / np.log(10) / (WAVELENGTH_UM * 1e-4)
# Device averages over the striated length: the segment's values times FILL.
loss_db_cm = {
    label: FILL * LOSS_DB_PER_CM_PER_IMAG * np.abs(n.imag) for label, n in n_eff.items()
}
index_shift = {
    label: FILL * (n.real - n.real[np.argmin(np.abs(graded_v))])
    for label, n in n_eff.items()
}
i0 = int(np.argmin(np.abs(graded_v)))
for label in n_eff:
    print(
        f"{label}: n_eff(0 V) = {n_eff[label][i0].real:.5f}, loss(0 V) = {loss_db_cm[label][i0]:.2f} dB/cm"
    )

For contrast, the as-stated profile of Step 1 goes through the same chain with
Soref-Bennett only.

In [ ]:
stated_n_eff = np.array(
    [
        solve_optical_mode(point.carriers, DISPERSION_MODELS["Soref-Bennett"])
        for point in stated_sweep.points
    ]
)
stated_v = stated_sweep.voltages
stated_index_shift = FILL * (
    stated_n_eff.real - stated_n_eff.real[np.argmin(np.abs(stated_v))]
)
stated_loss_db_cm = FILL * LOSS_DB_PER_CM_PER_IMAG * np.abs(stated_n_eff.imag)
print(
    f"as-stated profile, Soref-Bennett: loss(0 V) = {stated_loss_db_cm[np.argmin(np.abs(stated_v))]:.2f} dB/cm"
)

The left panel compares the simulated $V_\pi L$ with the paper's two arms. Look
for the simulation between or close to the arms from -0.1 V up. The right panel
shows $\Delta n_{eff}$ for the two dispersion models.

In [ ]:
from gsim.common.twmzm import vpi_length_vcm


def vpil_at(biases, label):
    """Local-slope V_pi L (V cm) at the given biases, from a spline through the sweep."""
    return np.array(
        [
            vpi_length_vcm(
                graded_v,
                index_shift[label],
                wavelength_um=WAVELENGTH_UM,
                bias_v=float(v),
            )
            for v in biases
        ]
    )


VPIL_PAPER = {arm: np.array(REF["vpil_V_cm"][arm]) for arm in ("top_arm", "bottom_arm")}
vpil_sim = {label: vpil_at(VPIL_BIASES, label) for label in n_eff}
paper_lo = np.minimum(*VPIL_PAPER.values())
paper_hi = np.maximum(*VPIL_PAPER.values())
mid = np.mean([paper_lo, paper_hi], axis=0)
# Reading uncertainty of the digitised curve: one marker, 0.1-0.2 V cm (reference
# file note); the middle of that range is used.
MARKER_VCM = 0.15
# Bias is "reverse" positive: the paper's text says positive voltage reverse-biases
# the junction, so negative bias is forward bias. The point singled out is the most
# forward one, at the end of the digitised curve (-0.4 V); -0.1 V is forward too,
# but by a quarter of that, and the two arms differ by 9% there against 15%.
FORWARD = np.isclose(VPIL_BIASES, VPIL_BIASES.min())
bar_lo, bar_hi = paper_lo - MARKER_VCM, paper_hi + MARKER_VCM

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.axvspan(VPIL_BIASES[0] - 0.15, 0.0, color="C1", alpha=0.08, label="forward bias")
ax.fill_between(
    VPIL_BIASES,
    paper_lo,
    paper_hi,
    color="k",
    alpha=0.12,
    label="spread between the two arms",
)
ax.errorbar(
    VPIL_BIASES[FORWARD],
    mid[FORWARD],
    yerr=[(mid - bar_lo)[FORWARD], (bar_hi - mid)[FORWARD]],
    fmt="none",
    ecolor="k",
    capsize=4,
    label="arm spread +/- one marker (forward-bias end)",
)
ax.plot(VPIL_BIASES, VPIL_PAPER["top_arm"], "ks", label="paper, top arm (read by eye)")
ax.plot(VPIL_BIASES, VPIL_PAPER["bottom_arm"], "k^", label="paper, bottom arm")
for (label, values), style in zip(vpil_sim.items(), ("o-", "s--"), strict=True):
    ax.plot(VPIL_BIASES, values, style, label=f"simulation, {label}")
stated_max = float(stated_v.max())
stated_biases = np.array([v for v in VPIL_BIASES if v <= stated_max])
vpil_stated = np.array(
    [
        vpi_length_vcm(
            stated_v, stated_index_shift, wavelength_um=WAVELENGTH_UM, bias_v=float(v)
        )
        for v in stated_biases
    ]
)
ax.plot(
    stated_biases,
    vpil_stated,
    ":",
    color="C0",
    alpha=0.7,
    label="simulation, doping as stated (Soref-Bennett)",
)
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("$V_\\pi L$ (V cm)")
ax.set_title("Modulation efficiency, Fig. 2c")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax = axes[1]
for label in n_eff:
    ax.plot(graded_v, index_shift[label] * 1e5, "o-", label=label)
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("$\\Delta n_{eff} \\times 10^{5}$")
ax.set_title("Effective-index shift")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

primary = "Soref-Bennett"
print("  V    paper top  paper bottom   sim(Soref)  sim(Nedeljkovic)   (V cm)")
for i, v in enumerate(VPIL_BIASES):
    print(
        f"{v:5.2f}   {VPIL_PAPER['top_arm'][i]:5.2f}      {VPIL_PAPER['bottom_arm'][i]:5.2f}"
        f"         {vpil_sim[primary][i]:5.2f}       {vpil_sim['Nedeljkovic'][i]:5.2f}"
    )
within_spread = (vpil_sim[primary] >= paper_lo * 0.9) & (
    vpil_sim[primary] <= paper_hi * 1.1
)
print(
    f"Soref points inside the arm spread +/-10%: {int(within_spread.sum())} of {len(VPIL_BIASES)}"
)
print(f"Soref / mid-arm: {np.round(vpil_sim[primary] / mid, 2)}")
print(
    f"points from {VPIL_BIASES[~FORWARD][0]:g} V up: worst error against the mid-arm value"
    f" {np.max(np.abs(vpil_sim[primary][~FORWARD] / mid[~FORWARD] - 1)):.1%}"
)
for k in np.flatnonzero(FORWARD):
    sim_v = vpil_sim[primary][k]
    print(
        f"end of the curve, {VPIL_BIASES[k]:g} V (forward bias): simulation {sim_v:.2f} V cm ({sim_v / mid[k] - 1:+.0%} from the mid-arm value);"
        f" paper arms {paper_lo[k]:.2f} to {paper_hi[k]:.2f}, with +/-{MARKER_VCM:g} V cm of reading"
        f" uncertainty {bar_lo[k]:.2f} to {bar_hi[k]:.2f}: {'inside' if bar_lo[k] <= sim_v <= bar_hi[k] else 'outside'}"
        f" by {max(bar_lo[k] - sim_v, sim_v - bar_hi[k], 0):.2f} V cm"
    )

From -0.1 V up, the Soref-Bennett $V_\pi L$ is within 10.7% of the mid-arm
value: 1.01 to 1.07 times up to 2.5 V, then 0.94 and 0.89 times at 3.5 and
4.5 V. Nine of the ten points lie inside the arm spread ±10%. The Nedeljkovic
model gives a $V_\pi L$ about 20% higher.

### The most forward point (-0.4 V)

The $V_\pi L$ at -0.4 V is the one point outside the ±25% band, and it is not
scored. The paper takes positive voltage as reverse bias ("positive voltage
would result in a reverse-bias"), so -0.4 V is a forward bias at the very end
of the digitised curve. There the two arms already differ by 15% and the curve
is steep, so a small error in reading the bias is a large one in $V_\pi L$.

Three checks follow:

1. **Numerics.** The slope at -0.4 V is an interior one, because the sweep runs
   to -0.5 V, and it does not depend on the estimator. The terminal current is
   per cm of device and tiny, so the solver is nowhere near injection. The
   paper reports below 10 µA up to 0.5 V forward.
2. **The paper against itself.** $V_\pi L \propto 1 / (d\Delta n_{eff}/dV)$ and
   the index change follows the depletion charge, so $V_\pi L \times C$ varies
   slowly with bias. The check evaluates the product from the paper's own
   Fig. 2c and 2d and from the simulation.
3. **Differential or chord.** The caption says small-signal, which is the local
   slope. The same $\Delta n_{eff}$ read as a chord from 0 V shows that the
   paper's data follow the local reading.

In [ ]:
from scipy.interpolate import CubicSpline

dn_device = index_shift[primary]
volts = graded_v


def vpil_from_slope(slope):
    """V_pi L (V cm) from d(dn_eff)/dV."""
    return WAVELENGTH_UM / (2 * abs(slope)) / 1e4


j_fwd = int(np.argmin(np.abs(volts + 0.4)))
inner = volts > -0.45  # without the -0.5 V point
estimates = {
    "cubic spline, whole sweep (used above)": CubicSpline(volts, dn_device)(-0.4, 1),
    "cubic spline, sweep from -0.4 V (edge slope)": CubicSpline(
        volts[inner], dn_device[inner]
    )(-0.4, 1),
    "natural spline, sweep from -0.4 V": CubicSpline(
        volts[inner], dn_device[inner], bc_type="natural"
    )(-0.4, 1),
    "central difference, -0.5 to -0.25 V": (dn_device[j_fwd + 1] - dn_device[j_fwd - 1])
    / (volts[j_fwd + 1] - volts[j_fwd - 1]),
}
print("V_pi L at -0.4 V by slope estimator (V cm):")
for name, slope in estimates.items():
    print(f"  {name:46s} {vpil_from_slope(slope):.3f}")
fwd = graded_sweep.points[j_fwd]
print(
    f"at -0.4 V: terminal current {abs(next(iter(fwd.currents_a_per_cm.values()))):.1e} A/cm,"
    f" C = {graded_ff_um[j_fwd]:.3f} fF/um"
)

In [ ]:
c_paper_at = np.interp(VPIL_BIASES, CV_BIASES, CV_PAPER_FF_UM)
c_sim_at = np.interp(VPIL_BIASES, graded_v, graded_ff_um)
usable = np.less_equal(VPIL_BIASES, CV_BIASES.max())  # Fig. 2d ends at 4 V
product_paper = (mid * c_paper_at)[usable]
product_sim = (vpil_sim[primary] * c_sim_at)[usable]
print("\n  V    V_pi L x C  paper   simulation   sim/paper   (V cm fF/um)")
for v, a, b in zip(VPIL_BIASES[usable], product_paper, product_sim, strict=True):
    print(f"{v:5.2f}   {a:10.3f}   {b:10.3f}   {b / a:8.2f}")
print(
    f"paper's own product rises {product_paper[1] / product_paper[0] - 1:+.0%} from -0.4 to -0.1 V,"
    f" the simulation's {product_sim[1] / product_sim[0] - 1:+.0%}"
)

In [ ]:
i_end = int(np.argmin(np.abs(graded_v - VPIL_BIASES[-1])))
chord = vpil_from_slope(dn_device[i_end] / graded_v[i_end])
print(
    f"at {VPIL_BIASES[-1]:g} V: differential reading {vpil_sim[primary][-1]:.2f} V cm,"
    f" chord reading {chord:.2f} V cm; paper {VPIL_PAPER['top_arm'][-1]:.2f} (top arm),"
    f" {VPIL_PAPER['bottom_arm'][-1]:.2f} (bottom arm)"
)
print(
    f"at -0.4 V: simulation {vpil_sim[primary][0]:.2f} V cm, paper top arm {VPIL_PAPER['top_arm'][0]:.2f},"
    f" bottom arm {VPIL_PAPER['bottom_arm'][0]:.2f}; the paper's curve there falls"
    f" {abs(VPIL_PAPER['top_arm'][1] - VPIL_PAPER['top_arm'][0]) / 0.3:.1f} V cm per V,"
    f" so a 0.05 V reading error is {0.05 * abs(VPIL_PAPER['top_arm'][1] - VPIL_PAPER['top_arm'][0]) / 0.3:.2f} V cm"
)

The four slope estimators give 1.36 to 1.43 V cm at -0.4 V, and the terminal
current is 1.4e-11 A per cm of device. The paper's own $V_\pi L \times C$
rises by 42% from -0.4 to -0.1 V (0.277 to 0.394 V cm fF/µm) where the
simulation's rises by 9%, and the simulation agrees with the paper's product
within 8% at every other bias. The chord reading at 4.5 V is 3.09 V cm against
the paper's 4.60 and 5.35, so the data follow the local slope.

At -0.4 V the paper's curve falls 2.2 V cm per V, so a 0.05 V error in reading
the bias is 0.11 V cm. The point is a limit of the data at the forward-bias
end. It does not confirm the model there.

### Group index

The electro-optic response in Step 3 needs the optical group index,
$n_g = n_{eff} - \lambda\, dn_{eff}/d\lambda$. A central difference over
±10 nm of wavelength at 0 V gives it, and the paper quotes 3.86.

In [ ]:
STEP_UM = 0.01  # wavelength step (um)
n_below, n_above = (
    solve_optical_mode(
        graded_sweep.points[i0].carriers,
        DISPERSION_MODELS[primary],
        WAVELENGTH_UM + step,
    ).real
    for step in (-STEP_UM, STEP_UM)
)
N_GROUP = n_eff[primary][i0].real - WAVELENGTH_UM * (n_above - n_below) / (2 * STEP_UM)
print(
    f"n_eff = {n_eff[primary][i0].real:.4f}, group index n_g = {N_GROUP:.4f} (paper: {REF['optical']['n_group']})"
)
print(
    f"  from n_eff = {n_below:.4f} / {n_above:.4f} at {WAVELENGTH_UM - STEP_UM:g} / {WAVELENGTH_UM + STEP_UM:g} um"
)
print(
    f"deviation from the paper: {N_GROUP / REF['optical']['n_group'] - 1:+.1%} (target +/-3%)"
)

The group index is 3.893, +0.9% from the paper's 3.86 (target ±3%).

### What the paper's 10 dB/cm is

The paper does not measure the implant loss directly. It reports an insertion
loss of 8.7 ± 0.7 dB for the Mach-Zehnder, "corresponding to an approximate
excess loss from the implants of 10 dB/cm", against its own prediction of
6.7 dB/cm. The y-junctions take approximately 2.6 dB, "with the rest of the
loss due to the waveguide itself". The figure is therefore the insertion loss
minus the y-junctions and an unstated waveguide loss, spread over the 5 mm
phase shifter.

The cell below does that arithmetic. The insertion-loss error alone is
±1.4 dB/cm, so **10 dB/cm is an estimate good to at least ±1.5 dB/cm**. The
comparison uses both the band 8.5 to 11.5 dB/cm and the paper's own prediction
of 6.7 dB/cm.

In [ ]:
PS_LENGTH_CM = REF["geometry"]["phase_shifter_length_um"] * 1e-4
MZI_LOSS_DB = REF["optical"]["mzi_insertion_loss_dB"]
MZI_LOSS_ERR_DB = REF["optical"]["mzi_insertion_loss_err_dB"]
Y_JUNCTION_DB = REF["optical"]["y_junction_loss_dB"]
LOSS_MEASURED = REF["optical"]["implant_loss_dB_per_cm_measured"]
LOSS_PREDICTED = REF["optical"]["implant_loss_dB_per_cm_paper_prediction"]
# dB/cm, "at least": the waveguide loss the paper subtracts is not stated.
LOSS_MEASURED_ERR = 1.5
LOSS_BAND = (LOSS_MEASURED - LOSS_MEASURED_ERR, LOSS_MEASURED + LOSS_MEASURED_ERR)

ceiling = (MZI_LOSS_DB - Y_JUNCTION_DB) / PS_LENGTH_CM
print(
    f"MZI insertion loss {MZI_LOSS_DB} +/- {MZI_LOSS_ERR_DB} dB, y-junctions {Y_JUNCTION_DB} dB, over {PS_LENGTH_CM * 10:g} mm:"
)
print(
    f"  before the waveguide loss: {ceiling:.1f} dB/cm; the insertion-loss error alone is"
    f" +/-{MZI_LOSS_ERR_DB / PS_LENGTH_CM:.1f} dB/cm"
)
print(
    f"  10 dB/cm implies a waveguide loss of {(ceiling - LOSS_MEASURED) * PS_LENGTH_CM:.1f} dB, which the paper does not state"
)
print(
    f"  -> used here: measured {LOSS_MEASURED:g} +/- {LOSS_MEASURED_ERR:g} dB/cm (band {LOSS_BAND[0]:g} to {LOSS_BAND[1]:g}),"
    f" the paper's own prediction {LOSS_PREDICTED} dB/cm"
)
print()
for label in n_eff:
    sim_loss = loss_db_cm[label][i0]
    print(
        f"{label}: implant loss at 0 V = {sim_loss:.2f} dB/cm:"
        f" {sim_loss / LOSS_MEASURED - 1:+.0%} from {LOSS_MEASURED:g}"
        f" ({sim_loss - LOSS_BAND[0]:+.1f} dB/cm from the low end of the band),"
        f" {sim_loss / LOSS_PREDICTED - 1:+.0%} from the prediction {LOSS_PREDICTED}"
    )
stated_loss_0v = stated_loss_db_cm[np.argmin(np.abs(stated_v))]
i_cv0 = int(np.argmin(np.abs(CV_BIASES)))
print(
    f"as-stated uniform profile, Soref-Bennett: {stated_loss_0v:.1f} dB/cm"
    f" ({stated_loss_0v / LOSS_MEASURED - 1:+.0%} from {LOSS_MEASURED:g}), but with"
    f" {stated_ff_um[i_cv0] / CV_PAPER_FF_UM[i_cv0]:.1f} times the paper's capacitance at 0 V"
)

### Checking the implant loss

The implant loss at 0 V is 4.1 dB/cm with Soref-Bennett and 3.2 dB/cm with
Nedeljkovic. That is 59% and 68% below the paper's 10 dB/cm estimate, and 38%
and 52% below its own prediction of 6.7 dB/cm. Before calling this a limit of
the model, the next cells check the chain independently.

The check does not use the complex $n_{eff}$. It integrates the absorbed power
of the same carrier map over the mode field, region by region,

$$\alpha = \frac{1}{2}\, c\, \varepsilon_0\, \frac{\int \alpha(\mathbf r)\, n(\mathbf r)\, |\mathbf E|^2 dA}{P_z},
\qquad P_z = \tfrac{1}{2}\int \mathrm{Re}(\mathbf E \times \mathbf H^*)_z\, dA,$$

and prints how much of the mode sits in each region and the carrier density it
sees there. Agreement with the $n_{eff}$ value tests the units, the
$10/\ln 10$ factor, the field-to-power factor of two and the TE0 mode.

In [ ]:
from skfem import Functional

C_LIGHT, EPS_0 = 299792458.0, 8.8541878128e-12


def loss_budget(point, dispersion):
    """Segment loss (dB/cm) from the field overlap, by region, and carriers seen by the mode."""
    mode, on_mesh, absorption_cm, epsilon = optical_mode(point.carriers, dispersion)
    region = np.asarray(on_mesh.region)
    basis, basis0 = mode.basis, mode.basis_epsilon_r
    e_field, h_field = basis.interpolate(mode.E), basis.interpolate(mode.H)
    index = basis0.interpolate(np.sqrt(epsilon.real))

    @Functional
    def flux(w):
        (ex, ey), _ = w["E"]
        (hx, hy), _ = w["H"]
        return 0.5 * np.real(ex * np.conj(hy) - ey * np.conj(hx))

    @Functional
    def absorbed(w):
        (ex, ey), ez = w["E"]
        return w["alpha"] * w["n"] * (abs(ex) ** 2 + abs(ey) ** 2 + abs(ez) ** 2)

    @Functional
    def energy(w):
        (ex, ey), ez = w["E"]
        return abs(ex) ** 2 + abs(ey) ** 2 + abs(ez) ** 2

    power = flux.assemble(basis, E=e_field, H=h_field)
    energy_per_element = energy.elemental(basis, E=e_field)
    rows = {}
    for name in REGIONS:
        mine = region == name
        alpha_m = np.where(mine, absorption_cm * 100.0, 0.0)  # 1/m
        integral = absorbed.assemble(
            basis, E=e_field, alpha=basis0.interpolate(alpha_m), n=index
        )
        loss = 0.5 * C_LIGHT * EPS_0 * integral / power * 1e-2 * 10 / np.log(10)
        weight = energy_per_element[mine]
        rows[name] = (
            loss,
            weight.sum() / energy_per_element.sum(),
            (on_mesh.electrons_cm3[mine] * weight).sum() / weight.sum(),
            (on_mesh.holes_cm3[mine] * weight).sum() / weight.sum(),
        )
    return complex(mode.n_eff), rows, mode.te_fraction

In [ ]:
budget = {}
for label, sweep_label in (("graded", graded_sweep), ("as stated", stated_sweep)):
    point = sweep_label.points[int(np.argmin(np.abs(sweep_label.voltages)))]
    n_complex, rows, te_fraction = loss_budget(point, DISPERSION_MODELS[primary])
    budget[label] = rows
    total = sum(r[0] for r in rows.values())
    print(
        f"{label} profile, {primary}, 0 V (segment values, TE fraction {te_fraction:.3f}):"
        f" overlap integral {total:.2f} dB/cm, from n_eff {LOSS_DB_PER_CM_PER_IMAG * abs(n_complex.imag):.2f} dB/cm"
    )
    print("  region       loss (dB/cm)  |E|^2 fraction   <n> (cm^-3)   <p> (cm^-3)")
    for name, (loss, share, mean_n, mean_p) in rows.items():
        print(
            f"  {name:10s} {loss:9.3f}   {share:12.3f}   {mean_n:11.2e}   {mean_p:11.2e}"
        )
    slab_loss = rows["n_slab"][0] + rows["p_slab"][0]
    slab_carriers = (rows["n_slab"][2] + rows["p_slab"][3]) / 2
    print(
        f"  both slabs {slab_loss:.2f} dB/cm at a mean {slab_carriers:.1e} cm^-3; at 1e18 cm^-3"
        f" (a hypothetical intermediate implant) about {slab_loss * 1e18 / slab_carriers:.1f} dB/cm"
    )

The overlap integral gives 4.65 dB/cm against 4.59 dB/cm from
$\mathrm{Im}\,n_{eff}$ for the same segment, so the chain is consistent. The
two rib halves carry 62% of the field and the two slabs 7%. The slabs add
0.5 dB/cm at their mean 1.8e17 cm⁻³, or 2.9 dB/cm if an intermediate implant
brought them to 1e18 cm⁻³, and the contacts add less than 0.01 dB/cm.

**What the mode sees.** The loss is carried by the carriers where the field
is, and the field fills the whole rib. The capacitance is set by the doping
where the depletion region ends, a thin layer at the junction. The table
compares the paper's quoted peak densities with the majority-carrier density
averaged over the mode ($|\mathbf E|^2$-weighted) in each rib half, for the
calibrated profile and for the profile as stated.

In [ ]:
PEAKS = {"n_rib": N_PEAK, "p_rib": P_PEAK}
print(
    "rib half   quoted peak   mode-weighted, calibrated (ratio)   mode-weighted, as stated (ratio)"
)
for name, peak in PEAKS.items():
    column = 2 if name == "n_rib" else 3  # electrons in the n side, holes in the p side
    graded_density = budget["graded"][name][column]
    stated_density = budget["as stated"][name][column]
    print(
        f"{name:9s}  {peak:9.1e}     {graded_density:9.2e} ({graded_density / peak:4.2f})"
        f"                     {stated_density:9.2e} ({stated_density / peak:4.2f})"
    )

For the calibrated profile the mode sees 6.3e16 cm⁻³ on the n side (0.13 of
the quoted 5e17) and 2.55e17 cm⁻³ on the p side (0.36 of the quoted 7e17). The
as-stated profile sees 0.79 and 0.85 of the quoted peaks.

### Loss against junction grading

A sharper junction (smaller σ) puts more carriers in the rib and raises both
the capacitance and the loss. The quoted peak doping is the plateau of every
profile in this family, so the sharpest junction is also the lossiest one the
family can give. The next cells repeat the chain for four values of σ at 0 V,
on the coarse mesh.

In [ ]:
SCAN_SIGMAS = (0.15, 0.20, SIGMA_UM, 0.35)

In [ ]:
scan = {}
for sigma in SCAN_SIGMAS:
    sim, _ = build_charge_sim(
        graded_profiles(sigma),
        output_dir=f"./tcad-bj2012/scan-{sigma:g}",
        refined_um=0.04,
        junction_box_um=0.02,
    )
    point = sim.sweep([0.0], contact="cathode").points[0]
    scan[sigma] = {
        "c_eff_ff_um": FILL * point.capacitance_f_per_cm * F_PER_CM_TO_FF_PER_UM,
        **{
            label: FILL
            * LOSS_DB_PER_CM_PER_IMAG
            * abs(solve_optical_mode(point.carriers, model).imag)
            for label, model in DISPERSION_MODELS.items()
        },
    }

The next figure plots the implant loss against the capacitance at 0 V, one
point per σ. Look at where the points fall against the grey band, which marks
the capacitance within 10% of Fig. 2d, and against the paper's loss band.

In [ ]:
c_paper_0v = float(np.interp(0.0, CV_BIASES, CV_PAPER_FF_UM))
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ratios = [scan[s]["c_eff_ff_um"] / c_paper_0v for s in SCAN_SIGMAS]
for label, style in zip(DISPERSION_MODELS, ("o-", "s--"), strict=True):
    ax.plot(ratios, [scan[s][label] for s in SCAN_SIGMAS], style, label=label)
for s, r in zip(SCAN_SIGMAS, ratios, strict=True):
    ax.annotate(
        f"sigma {s:g}",
        (r, scan[s][primary]),
        fontsize=7,
        xytext=(3, 4),
        textcoords="offset points",
    )
ax.axhspan(
    *LOSS_BAND,
    color="k",
    alpha=0.12,
    label=f"paper, estimated from the MZI: {LOSS_MEASURED:g} +/- {LOSS_MEASURED_ERR:g}",
)
ax.axhline(LOSS_MEASURED, color="k", ls="--", lw=0.8)
ax.axhline(LOSS_PREDICTED, color="k", ls=":", label="paper, its prediction")
ax.plot(
    stated_ff_um[i_cv0] / c_paper_0v,
    stated_loss_0v,
    "^",
    color="C0",
    label="doping as stated (uniform, abrupt), Soref-Bennett",
)
ax.axvspan(0.9, 1.1, color="0.9", label="C(0 V) within 10% of Fig. 2d")
ax.set_xlabel("C(0 V) / paper (device length)")
ax.set_ylabel("implant loss at 0 V (dB/cm, device average)")
ax.set_title("Loss against capacitance for different junction grading")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()
for s, r in zip(SCAN_SIGMAS, ratios, strict=True):
    print(
        f"sigma {s:.2f} um: C(0 V)/paper {r:.2f}, loss {scan[s][primary]:.1f} dB/cm ({primary}),"
        f" {scan[s]['Nedeljkovic']:.1f} (Nedeljkovic)"
    )

The calibrated profile (σ = 0.27 µm) gives 4.1 dB/cm with Soref-Bennett,
below both the 8.5 to 11.5 dB/cm band and the paper's own 6.7 dB/cm.
Sharpening the junction raises the loss from 3.3 dB/cm (σ = 0.35 µm) to
6.6 dB/cm (σ = 0.15 µm) but takes C(0 V) from 6% below to 14% above the
paper. The paper's own 6.7 dB/cm would need a capacitance at least 14% too
high.

**No profile in this family fits both the capacitance and the loss.** The
single erfc width σ is fitted to C(V), which sees the doping only at the edges
of the depletion region. The loss sees the whole mode, and for the calibrated
profile the mode-weighted densities are 6e16 cm⁻³ (n side) and 2.6e17 cm⁻³
(p side) against quoted peaks of 5e17 and 7e17. A profile with two parameters,
the junction grading and the width of a plateau at the quoted peaks, could
keep the junction lightly doped while the rest of the rib stays at the peaks.
It is not implemented here and its fit is untested.

## Step 3: loaded line and electro-optic bandwidth (closed form)

The electrode runs over the junction, so the junction loads it. Per unit
length the line's shunt admittance gains the series-RC branch
$j\omega C_j / (1 + j\omega R_s C_j)$, with $R_s$ and $C_j$ read from the
small-signal solve of Step 1 (`point.junction_branch()`). The branch corner,
$1/(2\pi R_s C_j)$, is 48 GHz at 0 V, so the 1 GHz fit frequency of that
solve is adequate. The loaded line follows from $\gamma = \sqrt{ZY}$ and
$Z_0 = \sqrt{Z/Y}$, and the electro-optic response from averaging the voltage
wave, with its reflections, along the co-propagating light.

**The metal-only line is the paper's own measurement.** At 10 GHz it has
$Z_0 = 61 - 5i\ \Omega$, 3.8 dB/cm, $R$ = 4.9 kΩ/m, $L$ = 457 nH/m and
$C$ = 0.12 nF/m. The telegrapher relations turn these into per-unit-length
$R, L, G, C$. The frequency dependence is an assumption: $R \propto \sqrt{f}$
(skin effect), with $L$, $C$ and $G$ constant. The
[Palace electrode notebook](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_bj2012_electrode.ipynb)
replaces this input by a solved electrode.

**Striations.** The junction branch from the solve is that of a segment. The
striated electrode (Fig. 1a inset) averages to a line with the branch
$(R_s/\mathrm{FILL},\ \mathrm{FILL}\cdot C_j)$: the same $R_s C_j$ time
constant, with the capacitance and the conductance per unit length scaled by
`FILL`. The striation check at the end of this step confirms this against the
periodic-line model.

**Termination and bandwidth.** The paper terminates each arm with 25 Ω
(GSSGSSG layout) and drives from 50 Ω. It states that "the primary bandwidth
limitation for electrooptic modulation is the RF loss" and that the 3 dB point
is reached "when there is around 6 dB of RF loss at the end of the
transmission line". That is a statement about the voltage that enters the
line, so the primary comparison references the response to the voltage at the
line input. The paper does not state the bias of the loaded-line measurement
(0 V is used), the probe and cable of the S21 measurement, or the frequency law
of the RF loss.

In [ ]:
from gsim.common.junction import JunctionBranch
from gsim.common.transmission_line import (
    bragg_fraction,
    line_params_from_gamma,
    loaded_line_params,
    rlgc_from_line_params,
    segmented_line,
)
from gsim.common.twmzm import eo_bandwidth, eo_response

metal = REF["metal_only_line_10GHz"]
loaded_paper = REF["loaded_line_10GHz"]
F_REF = metal["freq_ghz"] * 1e9
Z0_METAL = complex(metal["z0_real_ohm"], metal["z0_imag_ohm"])
R_REF = metal["r_kohm_per_m"] * 1e3
L_REF = metal["l_nH_per_m"] * 1e-9
NP_PER_M_TO_DB_PER_CM = 20 / np.log(10) * 1e-2

gamma_ref = (R_REF + 1j * 2 * np.pi * F_REF * L_REF) / Z0_METAL
rlgc_ref = rlgc_from_line_params([F_REF], gamma_per_m=[gamma_ref], z0_ohm=Z0_METAL)
RF_FREQS = np.linspace(0.5e9, 40e9, 80)  # contains 10 GHz
I_10GHZ = int(np.argmin(np.abs(RF_FREQS - F_REF)))


def metal_rlgc(freq_hz):
    """The metal-only line's R, L, G, C on a frequency grid (R follows sqrt(f))."""
    freq_hz = np.asarray(freq_hz, dtype=float)
    return {
        "R": R_REF * np.sqrt(freq_hz / F_REF),
        "L": np.full_like(freq_hz, rlgc_ref["L"][0]),
        "G": np.full_like(freq_hz, rlgc_ref["G"][0]),
        "C": np.full_like(freq_hz, rlgc_ref["C"][0]),
    }


def line_with(branch, freq_hz=RF_FREQS):
    """RFLineParams of the metal-only line loaded with a junction branch (None: unloaded)."""
    branch = JunctionBranch(0.0, 0.0) if branch is None else branch
    gamma, z0 = loaded_line_params(freq_hz, rlgc=metal_rlgc(freq_hz), junction=branch)
    return line_params_from_gamma(freq_hz, gamma, z0_ohm=z0)


def averaged_branch(branch, fill=FILL):
    """The striated electrode's averaged branch from a segment's branch."""
    return JunctionBranch(
        r_s_ohm_m=branch.r_s_ohm_m / fill, c_j_f_per_m=branch.c_j_f_per_m * fill
    )

The 0 V solve gives the segment's junction branch, and `averaged_branch` turns
it into the striated electrode's. The paper's own values are the comparison:
1.6 Ω cm of series resistance and 0.225 fF/µm of capacitance at 0 V.

In [ ]:
i_zero = int(np.argmin(np.abs(graded_v)))
branch_segment = graded_sweep.points[i_zero].junction_branch()
branch_tcad = averaged_branch(branch_segment)
branch_paper = JunctionBranch(
    r_s_ohm_m=REF["doping"]["rib_to_contact_resistance_ohm_cm"] * 1e-2,
    c_j_f_per_m=0.225e-9,
)
print(
    f"junction branch at 0 V, segment: R_s = {branch_segment.r_s_ohm_m * 100:.2f} ohm*cm, C_j = {branch_segment.c_j_f_per_m * 1e9:.3f} fF/um"
)
print(
    f"  device average (R_s / {FILL:g}, C_j x {FILL:g}): R_s = {branch_tcad.r_s_ohm_m * 100:.2f} ohm*cm, C_j = {branch_tcad.c_j_f_per_m * 1e9:.3f} fF/um"
    f" (corner {1 / (2 * np.pi * branch_tcad.r_s_ohm_m * branch_tcad.c_j_f_per_m) / 1e9:.0f} GHz)"
)
print("paper-quoted branch: R_s = 1.6 ohm*cm, C_j = 0.225 fF/um (Fig. 2d at 0 V)")

line_metal = line_with(None)
line_tcad = line_with(branch_tcad)
line_quoted = line_with(branch_paper)

The segment has $R_s$ = 1.35 Ω cm and $C_j$ = 0.247 fF/µm. Averaged over the
striations they become 1.50 Ω cm and 0.222 fF/µm, against the paper's 1.6 Ω cm
and 0.225 fF/µm, which favours the averaging.

The next figure shows the metal-only line and the line loaded with the
simulated and with the paper-quoted junction. Look for the loaded curves
passing through the red squares at 10 GHz.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
curves = (
    (line_metal, "k--", "metal only (paper's 10 GHz values)"),
    (line_tcad, "C3-", "loaded, TCAD junction at 0 V"),
    (line_quoted, "C0:", "loaded, paper-quoted junction"),
)
ghz = RF_FREQS / 1e9
for line, style, label in curves:
    axes[0].plot(ghz, line.z0_ohm.real, style, label=label)
    axes[1].plot(ghz, line.n_rf, style, label=label)
    axes[2].plot(ghz, line.alpha_rf_np_m * NP_PER_M_TO_DB_PER_CM, style, label=label)
axes[0].plot(10, metal["z0_real_ohm"], "ks", label="paper, metal only")
axes[0].plot(10, loaded_paper["z0_real_ohm"], "rs", label="paper, loaded")
axes[1].plot(10, loaded_paper["n_rf"], "rs", label="paper, loaded")
axes[2].plot(10, metal["alpha_dB_per_cm"], "ks", label="paper, metal only")
axes[2].plot(10, loaded_paper["alpha_dB_per_cm"], "rs", label="paper, loaded")
axes[0].set_ylabel("Re $Z_0$ (ohm)")
axes[1].set_ylabel("RF index $n_{RF}$")
axes[2].set_ylabel("RF loss (dB/cm)")
for ax in axes:
    ax.set_xlabel("frequency (GHz)")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("at 10 GHz         Re Z0   Im Z0    n_RF   alpha (dB/cm)")
for name, line in (
    ("metal only", line_metal),
    ("loaded, TCAD", line_tcad),
    ("loaded, quoted", line_quoted),
):
    z0 = line.z0_ohm[I_10GHZ]
    alpha_db = line.alpha_rf_np_m[I_10GHZ] * NP_PER_M_TO_DB_PER_CM
    print(
        f"{name:15s}  {z0.real:6.1f}  {z0.imag:6.1f}   {line.n_rf[I_10GHZ]:5.2f}   {alpha_db:6.2f}"
    )
print(
    f"{'paper, loaded':15s}  {loaded_paper['z0_real_ohm']:6.1f}  {loaded_paper['z0_imag_ohm']:6.1f}   "
    f"{loaded_paper['n_rf']:5.2f}   {loaded_paper['alpha_dB_per_cm']:6.2f}"
)

At 10 GHz the line loaded with the simulated junction has Re $Z_0$ = 36.9 Ω,
$n_{RF}$ = 3.72 and 10.3 dB/cm, against 37.0 Ω, 3.75 and 10.8 dB/cm in the paper
(-0.1%, -0.9% and -4.5%). The paper-quoted junction gives 36.9 Ω, 3.73 and
10.7 dB/cm, so the simulated junction does as well as the quoted one. The
metal-only line is 61.0 Ω, 2.26 and 3.60 dB/cm, and the loading lowers the
impedance and raises the index and the loss.

The next figure shows the same three quantities at 10 GHz against reverse
bias. Look at how they move away from the 0 V point, where the paper's values
sit; the paper does not state the bias of its measurement.

In [ ]:
loaded_vs_bias = []
for point in graded_sweep.points:
    line = line_with(averaged_branch(point.junction_branch()), RF_FREQS[[I_10GHZ]])
    loaded_vs_bias.append(
        (
            line.z0_ohm[0].real,
            line.n_rf[0],
            line.alpha_rf_np_m[0] * NP_PER_M_TO_DB_PER_CM,
        )
    )
loaded_vs_bias = np.array(loaded_vs_bias)
print("  V    Re Z0   n_RF   alpha (dB/cm)")
for k in (0, i_zero, len(graded_v) - 1):
    z0_re, n_rf, alpha_db = loaded_vs_bias[k]
    print(f"{graded_v[k]:5.2f}  {z0_re:6.1f}  {n_rf:5.2f}  {alpha_db:6.2f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, column, label, target in zip(
    axes,
    loaded_vs_bias.T,
    ("Re $Z_0$ (ohm)", "$n_{RF}$", "RF loss (dB/cm)"),
    (
        loaded_paper["z0_real_ohm"],
        loaded_paper["n_rf"],
        loaded_paper["alpha_dB_per_cm"],
    ),
    strict=True,
):
    ax.plot(graded_v, column, "o-", color="C3", label="TCAD junction")
    ax.plot(0.0, target, "rs", label="paper (bias not stated)")
    ax.set_xlabel("Reverse bias (V)")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.suptitle("Loaded line at 10 GHz against bias")
plt.tight_layout()
plt.show()

From -0.5 V to 4.5 V the loaded line goes from Re $Z_0$ = 33.9 Ω to 42.1 Ω,
from $n_{RF}$ = 4.02 to 3.28 and from 13.9 to 6.9 dB/cm of RF loss, with
36.9 Ω, 3.72 and 10.3 dB/cm at 0 V. Reverse bias lowers the junction
capacitance and unloads the line.

### Electro-optic response

The 5 mm electrode with the optical group index of Step 2 is evaluated in four
set-ups. They differ only in the termination and in the quantity the response
is referenced to:

- **25 Ω per-arm termination, referenced to the voltage at the line input**
  (an ideal voltage source, $Z_g \to 0$, normalised to its low-frequency
  value). This is the primary comparison. It is the quantity in the paper's
  rule that the 3 dB point is reached at about 6 dB of RF loss, which the cell
  after the figure checks.
- **The same 25 Ω load driven from 50 Ω, referenced to the source EMF** (a
  VNA-type source whose incident wave is held constant; at low frequency the
  drive divides as $Z_L/(Z_L+Z_g)$). This illustrates termination effects and
  is not a comparison with the paper: a measured S21 would need the probe and
  cable de-embedded to refer it to the source, which the paper does not
  describe.
- **A matched load** $|Z_0|$, with no reflections for any source.
- **A 50 Ω load.**

The paper also reports ripple in the S21, attributed to the mismatch between
line and terminator and to cross-talk between the arms. Neither is in this
one-line model.

In the next figure, look for where each curve crosses the dotted -3 dB line
against the paper's 10 GHz marker.

In [ ]:
LENGTH_M = REF["geometry"]["phase_shifter_length_um"] * 1e-6
Z_LOAD = REF["electro_optic"]["termination_ohm_per_arm"]
Z_GEN = 50.0  # ohm, the paper's drive: "0.63 Vpp at 50 ohm"
Z_IDEAL = 1e-9  # ohm, ideal voltage source: the line input voltage is the reference
EO_FREQS = np.linspace(0.1e9, 40e9, 400)
line_eo = line_with(branch_tcad, EO_FREQS)


def eo_curve(z_load, z_gen=Z_GEN, line=line_eo, n_opt=N_GROUP):
    """Normalised EO response and its 3 dB bandwidth (Hz) for a termination."""
    response = eo_response(
        EO_FREQS,
        length_m=LENGTH_M,
        n_rf=line.n_rf,
        n_opt=n_opt,
        alpha_rf_np_m=line.alpha_rf_np_m,
        z0_ohm=line.z0_ohm,
        z_load_ohm=z_load,
        z_gen_ohm=z_gen,
    )
    return response, eo_bandwidth(EO_FREQS, response)


z_line = float(np.abs(line_eo.z0_ohm[np.argmin(np.abs(EO_FREQS - F_REF))]))
INPUT_REF = f"{Z_LOAD:g} ohm, referenced to the line input (primary)"
SOURCE_REF = f"{Z_LOAD:g} ohm, 50 ohm source, source EMF (illustration)"
terminations = {
    INPUT_REF: (Z_LOAD, Z_IDEAL),
    SOURCE_REF: (Z_LOAD, Z_GEN),
    f"{z_line:.0f} ohm (= |Z0|, matched)": (z_line, Z_GEN),
    "50 ohm, 50 ohm source": (50.0, Z_GEN),
}
fig, ax = plt.subplots(figsize=(7, 4.2))
bandwidth = {}
for (label, (z_load, z_gen)), style in zip(
    terminations.items(), ("C3-", "C3:", "C2--", "C0:"), strict=True
):
    response, f3db = eo_curve(z_load, z_gen)
    bandwidth[label] = f3db
    ax.plot(
        EO_FREQS / 1e9,
        20 * np.log10(np.abs(response)),
        style,
        label=f"{label}: {f3db / 1e9:.1f} GHz",
    )
ax.axhline(20 * np.log10(1 / np.sqrt(2)), color="k", ls=":", lw=0.8)
ax.axvline(
    REF["electro_optic"]["f_3dB_GHz"],
    color="k",
    ls="--",
    label="paper: about 10 GHz (Fig. 3)",
)
ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("EO response (dB)")
ax.set_title("Electro-optic response of the 5 mm loaded electrode")
ax.set_ylim(-12, 1)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

F_3DB_INPUT_GHZ = bandwidth[INPUT_REF] / 1e9
F_3DB_SOURCE_GHZ = bandwidth[SOURCE_REF] / 1e9
for label, f3db in bandwidth.items():
    print(f"3 dB bandwidth, {label}: {f3db / 1e9:.1f} GHz")
print(f"paper: about {REF['electro_optic']['f_3dB_GHz']:.0f} GHz")

The paper's 25 Ω termination, referenced to the line input, gives a 3 dB
bandwidth of 10.8 GHz, 8% above the paper's 10 GHz. The same load driven from
50 Ω and referenced to the source EMF gives 19.0 GHz, a matched load 12.1 GHz
and a 50 Ω load 5.8 GHz.

### The paper's 6 dB rule

For a velocity-matched line, matched at the far end and referenced to the
voltage at its input, the RF voltage seen by the light is $e^{-\alpha z}$
averaged over the length $L$: $|H| = (1 - e^{-x})/x$ with $x = \alpha L$ the
one-way amplitude loss in Np. The 3 dB electro-optic point $|H| = 1/\sqrt 2$
is at $x \approx 0.74$ Np, about 6.4 dB of RF loss at the end of the line: the
paper's "around 6 dB".

The next cell solves the rule, checks the library's `eo_response` against it
and asks where the simulated line reaches that loss. The paper's measured
10.8 dB/cm over 5 mm is 5.4 dB at 10 GHz, so its own numbers satisfy the rule.

In [ ]:
NP_TO_DB = 20 / np.log(10)


def matched_line_response(x_np):
    """|(1 - exp(-x)) / x|: velocity-matched, matched line, x Np of one-way loss."""
    return (1 - np.exp(-x_np)) / x_np


X_RULE_NP = brentq(lambda x: matched_line_response(x) - 1 / np.sqrt(2), 0.1, 3.0)
X_RULE_DB = X_RULE_NP * NP_TO_DB
print(
    f"(1 - exp(-x))/x = 1/sqrt(2) at x = {X_RULE_NP:.3f} Np = {X_RULE_DB:.2f} dB of RF loss at the end of the line"
)
paper_loss_end_db = loaded_paper["alpha_dB_per_cm"] * LENGTH_M * 100
print(
    f"paper's loaded line, 10 GHz: {loaded_paper['alpha_dB_per_cm']} dB/cm over {LENGTH_M * 1e3:g} mm = {paper_loss_end_db:.1f} dB,"
    f" response {20 * np.log10(matched_line_response(paper_loss_end_db / NP_TO_DB)):.2f} dB"
)

# eo_response for a matched, velocity-matched line with this loss: flat at 1/sqrt(2).
flat = eo_response(
    EO_FREQS,
    length_m=LENGTH_M,
    n_rf=N_GROUP,
    n_opt=N_GROUP,
    alpha_rf_np_m=X_RULE_NP / LENGTH_M,
    z0_ohm=37.0,
    z_load_ohm=37.0,
    z_gen_ohm=Z_IDEAL,
)
print(
    f"eo_response on that line: |H| from {np.abs(flat).min():.6f} to {np.abs(flat).max():.6f}"
    f" (formula {matched_line_response(X_RULE_NP):.6f})"
)

# The simulated loaded line: RF loss at the end of the line against frequency.
loss_end_db = line_eo.alpha_rf_np_m * LENGTH_M * NP_TO_DB
assert np.all(np.diff(loss_end_db) > 0)
F_RULE_GHZ = float(np.interp(X_RULE_DB, loss_end_db, EO_FREQS)) / 1e9
print(
    f"simulated loaded line: {loss_end_db[np.argmin(np.abs(EO_FREQS - F_REF))]:.1f} dB at 10 GHz;"
    f" it reaches {X_RULE_DB:.1f} dB at {F_RULE_GHZ:.1f} GHz"
    f" (3 dB point, line-input reference: {F_3DB_INPUT_GHZ:.1f} GHz, with {np.interp(F_3DB_INPUT_GHZ * 1e9, EO_FREQS, loss_end_db):.1f} dB of RF loss)"
)

The simulated line has 5.2 dB of RF loss at 10 GHz and reaches 6.4 dB at
12.1 GHz. Its 25 Ω-loaded 3 dB point, 10.8 GHz, has 5.6 dB of RF loss.
`eo_response` returns a flat 0.707107 for a line with that loss, as the
formula does.

### Why the set-ups differ, and two checks

**(1) The response formula** (voltage wave with load and source reflections,
averaged over the co-propagating light) is checked against a brute-force
calculation that shares no code with it. The calculation steps exact line
sections back from the load, applies the source condition at the input and
integrates the voltage numerically along the light.

**(2) The paper's own loaded-line numbers** ($Z_0 = 37\ \Omega$,
$n_{RF} = 3.75$, 10.8 dB/cm at 10 GHz, 5 mm, $n_g = 3.86$) go through the same
formula, with the loss rising as $f^p$. This needs no TCAD. The 10.8 dB/cm over
5 mm is already 5.4 dB one way, so a matched line has lost 2.6 dB of
electro-optic response at 10 GHz whatever $p$ is.

With a 25 Ω load on a 37 Ω line driven from 50 Ω, the voltage that reaches the
line rises with frequency, because the line impedance replaces the load's in
the divider (+2.1 dB). That hides the loss when the response is referenced to
the source EMF, and is why the illustration case lands near 19 GHz. It is a
different quantity from the one in the paper's 6 dB rule, and comparing it with
a measured S21 would first need the probe and cable removed, which the paper
does not describe.

In [ ]:
def eo_brute_force(freq_hz, gamma, z0, z_load, z_gen, n_opt, length_m, n_steps=4001):
    """V_eff per volt of source EMF, DC-normalised, by stepping exact ABCD sections from the load."""
    z = np.linspace(0.0, length_m, n_steps)
    step = z[1] - z[0]
    cosh, sinh = np.cosh(gamma * step), np.sinh(gamma * step)
    v = np.empty(n_steps, dtype=complex)
    i = np.empty(n_steps, dtype=complex)
    v[-1], i[-1] = 1.0, 1.0 / z_load
    for k in range(n_steps - 2, -1, -1):
        v[k] = cosh * v[k + 1] + z0 * sinh * i[k + 1]
        i[k] = sinh / z0 * v[k + 1] + cosh * i[k + 1]
    v /= v[0] + i[0] * z_gen  # source EMF of 1 V
    light = np.exp(1j * 2 * np.pi * freq_hz * n_opt / C_LIGHT * z)
    averaged = np.sum((v * light)[1:] + (v * light)[:-1]) * step / 2 / length_m
    return averaged / (z_load / (z_load + z_gen))


worst = 0.0
for z_load, z_gen in ((Z_LOAD, Z_GEN), (Z_LOAD, Z_IDEAL), (z_line, Z_GEN)):
    response, _ = eo_curve(z_load, z_gen)
    for k in (4, 40, 100, 200, 300):
        gamma = (
            line_eo.alpha_rf_np_m[k]
            + 1j * 2 * np.pi * EO_FREQS[k] * line_eo.n_rf[k] / C_LIGHT
        )
        brute = eo_brute_force(
            EO_FREQS[k],
            gamma,
            line_eo.z0_ohm[k],
            z_load,
            max(z_gen, 1e-9),
            N_GROUP,
            LENGTH_M,
        )
        worst = max(worst, abs(brute - response[k]) / abs(response[k]))
print(
    f"response formula against the brute-force integration: largest relative difference {worst:.1e}"
)

In [ ]:
paper_f = np.linspace(0.1e9, 60e9, 600)
paper_alpha_np_m = loaded_paper["alpha_dB_per_cm"] / NP_PER_M_TO_DB_PER_CM  # at 10 GHz
print("the paper's own loaded-line numbers, 3 dB bandwidth (GHz); loss ~ f^p")
print("   p   25 ohm, line-input ref.   25 ohm, 50 ohm source   37 ohm (matched)")
for p in (1.0, 2.0):
    cells = []
    for z_load, z_gen in ((Z_LOAD, Z_IDEAL), (Z_LOAD, Z_GEN), (37.0, Z_GEN)):
        trial = eo_response(
            paper_f,
            length_m=LENGTH_M,
            n_rf=loaded_paper["n_rf"],
            n_opt=REF["optical"]["n_group"],
            alpha_rf_np_m=paper_alpha_np_m * (paper_f / 10e9) ** p,
            z0_ohm=complex(loaded_paper["z0_real_ohm"], loaded_paper["z0_imag_ohm"]),
            z_load_ohm=z_load,
            z_gen_ohm=z_gen,
        )
        cells.append(eo_bandwidth(paper_f, trial) / 1e9)
    print(f"{p:4.0f}   {cells[0]:21.1f}   {cells[1]:21.1f}   {cells[2]:15.1f}")

The formula agrees with the brute-force integration to 5e-8. For the paper's
own numbers, the line-input reference gives 10.6 GHz (p = 1) and 10.4 GHz
(p = 2), against 21.3 and 14.6 GHz for the 50 Ω source and 11.9 and 10.9 GHz
for a matched 37 Ω load. The paper's numbers alone therefore also put the
primary comparison at about 10 GHz.

### Striations

The paper's implants are segmented along the propagation direction (10 µm
period, 1 µm gaps, fill 0.9), so only 90% of the length carries the loading
capacitance. A segmented line is a periodic structure of loaded and unloaded
sections. It behaves as the length-weighted average line while the period stays
far below the Bragg condition. The check builds the periodic line from the
segment's branch and compares it with the averaged branch used above.

In [ ]:
PERIOD_M = REF["geometry"]["implant_striation_period_um"] * 1e-6
line_open = line_with(None, EO_FREQS)
line_segment = line_with(branch_segment, EO_FREQS)
line_seg = segmented_line(line_segment, line_open, fill_factor=FILL, period_m=PERIOD_M)
bragg = bragg_fraction(
    gamma_loaded_per_m=line_segment.gamma_per_m,
    gamma_unloaded_per_m=line_open.gamma_per_m,
    fill_factor=FILL,
    period_m=PERIOD_M,
)
i_ref = int(np.argmin(np.abs(EO_FREQS - F_REF)))
for name, line in (("averaged branch", line_eo), ("periodic line", line_seg)):
    alpha_db = line.alpha_rf_np_m[i_ref] * NP_PER_M_TO_DB_PER_CM
    print(
        f"{name:16s} at 10 GHz: Z0 = {line.z0_ohm[i_ref]:.2f} ohm,"
        f" n_RF = {line.n_rf[i_ref]:.3f}, alpha = {alpha_db:.2f} dB/cm"
    )
f3db_seg = {}
for set_up, z_gen in (("line-input reference", Z_IDEAL), ("50 ohm source", Z_GEN)):
    response_seg = eo_response(
        EO_FREQS,
        length_m=LENGTH_M,
        n_rf=line_seg.n_rf,
        n_opt=N_GROUP,
        alpha_rf_np_m=line_seg.alpha_rf_np_m,
        z0_ohm=line_seg.z0_ohm,
        z_load_ohm=Z_LOAD,
        z_gen_ohm=z_gen,
    )
    f3db_seg[set_up] = eo_bandwidth(EO_FREQS, response_seg)
print(
    f"largest Bragg fraction up to {EO_FREQS[-1] / 1e9:.0f} GHz: {bragg.max():.3f} (far below 0.05: a homogeneous-line model holds)"
)
for set_up, f_averaged in (
    ("line-input reference", F_3DB_INPUT_GHZ),
    ("50 ohm source", F_3DB_SOURCE_GHZ),
):
    print(
        f"3 dB bandwidth, 25 ohm load, {set_up}: periodic line {f3db_seg[set_up] / 1e9:.1f} GHz,"
        f" averaged branch {f_averaged:.1f} GHz"
    )

The periodic line and the averaged branch agree at 10 GHz (36.95 − 0.66i Ω,
$n_{RF}$ = 3.718, 10.31 dB/cm) and in the 3 dB bandwidths (10.8 GHz and
19.0 GHz). The largest Bragg fraction up to 40 GHz is 0.009, far below 0.05, so
a homogeneous-line model holds.

## Summary

The next cell lists the target (the paper), the simulation, the relative error
and the tolerance for every quantity of the notebook, and plots the errors.
Rows marked **fit** were adjusted to the target (one number, σ) and only show
that the fit worked. All other rows are predictions. The tolerances follow the
digitisation accuracy of the figures (about one marker) and what the paper
leaves unstated.

Two rows are not scored and are drawn with hollow markers: $V_\pi L$ at -0.4 V
(forward bias, the end of the digitised curve, shown against its error bar) and
the 50 Ω source case of the electro-optic response (an illustration of
termination effects, not a quantity the paper reports). The implant loss is
scored twice, against the band estimated from the Mach-Zehnder insertion loss
(8.5 to 11.5 dB/cm) and against the paper's own prediction. In the figure, a
green dot inside its grey band passes. Look for the red dots.

In [ ]:
vpil_ratio = vpil_sim[primary] / mid
k_fwd = int(np.flatnonzero(FORWARD)[0])  # the -0.4 V point
sim_fwd = vpil_sim[primary][k_fwd]
miss_fwd = max(bar_lo[k_fwd] - sim_fwd, sim_fwd - bar_hi[k_fwd], 0.0)
fwd_note = f"arms +/-{MARKER_VCM:g} V cm bar: " + (
    "inside" if miss_fwd == 0 else f"outside by {miss_fwd:.2f} V cm"
)
rows = [
    # name, kind, target, simulation, relative error, (lowest, highest) tolerated
    # error, scored?, note
    (
        "C(V), worst of 10 biases",
        "fit",
        "Fig.2d",
        "-",
        float(np.max(np.abs(ratio - 1))),
        (-0.10, 0.10),
        True,
        "",
    ),
    (
        "V_pi L, -0.1 V and up",
        "pred.",
        "Fig.2c",
        "-",
        float(np.max(np.abs(vpil_ratio[~FORWARD] - 1))),
        (-0.25, 0.25),
        True,
        "",
    ),
    (
        "V_pi L at -0.4 V (forward bias)",
        "pred.",
        mid[k_fwd],
        sim_fwd,
        float(vpil_ratio[k_fwd] - 1),
        (bar_lo[k_fwd] / mid[k_fwd] - 1, bar_hi[k_fwd] / mid[k_fwd] - 1),
        False,
        fwd_note,
    ),
    (
        "group index",
        "pred.",
        REF["optical"]["n_group"],
        N_GROUP,
        N_GROUP / REF["optical"]["n_group"] - 1,
        (-0.03, 0.03),
        True,
        "",
    ),
    (
        "implant loss vs MZI estimate (dB/cm)",
        "pred.",
        LOSS_MEASURED,
        loss_db_cm[primary][i0],
        loss_db_cm[primary][i0] / LOSS_MEASURED - 1,
        (LOSS_BAND[0] / LOSS_MEASURED - 1, LOSS_BAND[1] / LOSS_MEASURED - 1),
        True,
        f"band {LOSS_BAND[0]:g}-{LOSS_BAND[1]:g}",
    ),
    (
        "implant loss vs paper's prediction",
        "pred.",
        LOSS_PREDICTED,
        loss_db_cm[primary][i0],
        loss_db_cm[primary][i0] / LOSS_PREDICTED - 1,
        (1 / 1.5 - 1, 0.5),
        True,
        "factor 1.5",
    ),
    (
        "loaded Re Z0 (ohm)",
        "pred.",
        loaded_paper["z0_real_ohm"],
        line_tcad.z0_ohm[I_10GHZ].real,
        line_tcad.z0_ohm[I_10GHZ].real / loaded_paper["z0_real_ohm"] - 1,
        (-0.10, 0.10),
        True,
        "",
    ),
    (
        "loaded RF index",
        "pred.",
        loaded_paper["n_rf"],
        line_tcad.n_rf[I_10GHZ],
        line_tcad.n_rf[I_10GHZ] / loaded_paper["n_rf"] - 1,
        (-0.05, 0.05),
        True,
        "",
    ),
    (
        "loaded RF loss (dB/cm)",
        "pred.",
        loaded_paper["alpha_dB_per_cm"],
        line_tcad.alpha_rf_np_m[I_10GHZ] * NP_PER_M_TO_DB_PER_CM,
        line_tcad.alpha_rf_np_m[I_10GHZ]
        * NP_PER_M_TO_DB_PER_CM
        / loaded_paper["alpha_dB_per_cm"]
        - 1,
        (-0.25, 0.25),
        True,
        "",
    ),
    (
        "EO f3dB, 25 ohm, line-input ref.",
        "pred.",
        REF["electro_optic"]["f_3dB_GHz"],
        F_3DB_INPUT_GHZ,
        F_3DB_INPUT_GHZ / REF["electro_optic"]["f_3dB_GHz"] - 1,
        (7 / 10 - 1, 14 / 10 - 1),
        True,
        "paper's 6 dB rule convention",
    ),
    (
        "EO f3dB, 25/50 ohm, source EMF",
        "illus.",
        REF["electro_optic"]["f_3dB_GHz"],
        F_3DB_SOURCE_GHZ,
        F_3DB_SOURCE_GHZ / REF["electro_optic"]["f_3dB_GHz"] - 1,
        (7 / 10 - 1, 14 / 10 - 1),
        False,
        "termination effect; needs probe/cable de-embedding",
    ),
]


def fmt(value):
    return (
        f"{value:8.3f}" if isinstance(value, float | np.floating) else f"{value!s:>8}"
    )


def verdict(error, low, high, scored):
    """'ok' or 'OUTSIDE' for a scored row, 'not scored' otherwise."""
    if not scored:
        return "not scored"
    return "ok" if low <= error <= high else "OUTSIDE"


print(
    f"{'quantity':36s} {'kind':6s} {'target':>8s} {'sim':>8s} {'error':>8s}  {'tolerance':>14s}  result"
)
for name, kind, target, sim_value, error, (low, high), scored, note in rows:
    print(
        f"{name:36s} {kind:6s} {fmt(target)} {fmt(sim_value)} {error:+8.1%}  {low:+6.0%} .. {high:+5.0%}"
        f"  {verdict(error, low, high, scored)}{f' ({note})' if note else ''}"
    )

fig, ax = plt.subplots(figsize=(9, 4.6))
for k, (_name, _kind, _target, _sim, error, (low, high), scored, _note) in enumerate(
    rows
):
    ok = low <= error <= high
    ax.plot(
        [low * 100, high * 100], [k, k], "-", color="0.75", lw=8, solid_capstyle="butt"
    )
    color = "C2" if ok else "C3"
    ax.plot(
        error * 100,
        k,
        "o",
        color=color,
        markerfacecolor=color if scored else "none",
        markeredgewidth=1.5,
    )
ax.set_yticks(range(len(rows)), [r[0] for r in rows])
ax.invert_yaxis()
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel(
    "error against the paper (%); grey band = tolerance (or error bar); hollow = not scored"
)
ax.set_title("Simulation against the paper")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

Every scored row is inside its tolerance except the two implant-loss rows. The
C(V) fit is within 3.1% of Fig. 2d, and the predictions of Steps 2 and 3 are
within their tolerances apart from the loss.

| Quantity | Paper | Simulation | Status |
|---|---|---|---|
| $C(V)$, worst of 10 biases | Fig. 2d | within 3.1% | fit |
| $V_\pi L$, -0.1 V and up | Fig. 2c | within 10.7% of the mid-arm value | prediction, inside ±25% |
| $V_\pi L$ at -0.4 V (forward bias) | 1.03 V cm (mid-arm) | 1.35 V cm (+32%) | not scored |
| Group index | 3.86 | 3.89 (+0.9%) | prediction, inside ±3% |
| Implant loss at 0 V | 10 ± 1.5 dB/cm (insertion-loss estimate), 6.7 dB/cm (paper's prediction) | 4.1 dB/cm | prediction, outside both |
| Loaded line at 10 GHz: Re $Z_0$, $n_{RF}$, loss | 37.0 Ω, 3.75, 10.8 dB/cm | 36.9 Ω, 3.72, 10.3 dB/cm | prediction, inside tolerance |
| Electro-optic 3 dB bandwidth, 25 Ω, line-input reference | about 10 GHz | 10.8 GHz (+8%) | prediction, inside tolerance |
| Electro-optic 3 dB bandwidth, 25 Ω, 50 Ω source | not reported | 19.0 GHz | illustration, not scored |

### Limits of the comparison

- **The capacitance is fitted.** The paper does not give the doping profile,
  and the quoted peak doping as a uniform abrupt profile gives 1.5 to 2.0 times
  the measured capacitance (Step 1). That Fig. 2d is a device-length average is
  an assumption, and the striation fill applies it to every per-length
  quantity. Without the fill, the loss and $V_\pi L$ would change by about 10%.
- **The implant loss is a model limit.** The units, the $10/\ln 10$ factor and
  the TE0 mode are checked: the overlap integral gives 4.65 dB/cm against
  4.59 dB/cm from $\mathrm{Im}\,n_{eff}$. The one-parameter profile cannot fit
  both the capacitance and the loss. The as-stated profile gives 13.9 dB/cm,
  above the band, at 1.9 times the capacitance, and sharpening σ reaches
  6.6 dB/cm at a C(0 V) 14% too high. The target is soft as well: the
  10 dB/cm is the insertion loss minus the y-junctions and an unstated
  waveguide loss, 12.2 dB/cm before the waveguide term, with ±1.4 dB/cm from
  the insertion-loss error alone. A two-parameter profile is the next step and
  is not implemented. The 2D free-carrier model cannot say anything about the
  real implant shape, intermediate implants, or non-carrier loss such as
  sidewall scattering and implant damage.
- **$V_\pi L$ at -0.4 V is not scored.** It is a forward bias at the end of
  the digitised curve, where the two arms differ by 15% (1.10 and 0.95 V cm).
  The simulation gives 1.35 V cm. The error bar of ±0.15 V cm for one marker
  runs from 0.80 to 1.25 V cm, so the point is 0.10 V cm outside it (0.05 V cm
  with a 0.2 V cm marker). The numerics are not the cause, and the paper's own
  $V_\pi L \times C$ jumps by 42% between -0.1 and -0.4 V where the
  simulation's rises by 9%. The data at the forward-bias end limit the
  comparison there.
- **The bandwidth agrees under the paper's own convention.** The line-input
  reference matches the paper's 6 dB rule. The 19.0 GHz of the 50 Ω source case
  is an illustration, not a miss: the source-referenced response needs the
  probe and cable de-embedded, which the paper does not describe. The paper's
  ripple is not in the model, and the frequency law of the RF loss
  ($R \propto \sqrt{f}$) and the bias of the loaded-line measurement are
  assumptions.
- **The digitised values carry about one marker of uncertainty**: 0.01 fF/µm in
  the capacitance and 0.1 to 0.2 V cm in $V_\pi L$.